In [0]:
dbutils.widgets.removeAll()

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    IntegerType,
    LongType,
    DateType,
    TimestampType
)
from delta.tables import DeltaTable
from datetime import datetime
import re
import uuid

In [0]:
dbutils.widgets.text("catalogo", "catalog_au")
dbutils.widgets.text("esquema", "bronze")
dbutils.widgets.text("tabla_raw", "sfc_indicadores_raw")
dbutils.widgets.text("tabla_control", "sfc_archivos_control")
dbutils.widgets.text("volumen", "vol_sfc_raw")

In [0]:
catalogo = dbutils.widgets.get("catalogo")
esquema = dbutils.widgets.get("esquema")

tabla_raw = (
    f"{catalogo}.{esquema}."
    f"{dbutils.widgets.get('tabla_raw')}"
)

tabla_control = (
    f"{catalogo}.{esquema}."
    f"{dbutils.widgets.get('tabla_control')}"
)

ruta_origen = (
    f"/Volumes/{catalogo}/{esquema}/"
    f"{dbutils.widgets.get('volumen')}/"
)

archivos = [
    archivo
    for archivo in dbutils.fs.ls(ruta_origen)
    if archivo.name.lower().endswith(".xlsx")
    and not archivo.name.startswith("~$")
]

print("Tabla RAW:", tabla_raw)
print("Tabla CONTROL:", tabla_control)
print("Ruta origen:", ruta_origen)
print("Archivos encontrados:", len(archivos))

for archivo in archivos:
    print(archivo.name, archivo.size, "bytes")

In [0]:
df_archivos = (
    spark.table(tabla_raw)
    .groupBy("archivo_origen", "anio_origen")
    .agg(
        F.count("*").alias("total_celdas"),
        F.max("fila_excel").alias("total_filas"),
        F.max("columna_excel").alias("total_columnas"),
        F.min("fecha_ingesta").alias("fecha_inicio"),
        F.max("fecha_ingesta").alias("fecha_fin")
    )
)

display(df_archivos)

In [0]:
# Diccionario de archivos encontrados en ADLS
metadata_archivos = {
    archivo.name: {
        "ruta": archivo.path,
        "tamanio": archivo.size
    }
    for archivo in archivos
}

registros_control = []

for fila in df_archivos.collect():

    nombre = fila["archivo_origen"]

    if nombre not in metadata_archivos:
        print(f"ADVERTENCIA: Archivo no encontrado: {nombre}")
        continue

    metadata = metadata_archivos[nombre]

    # Extraer año y mes del nombre
    coincidencia = re.fullmatch(
        r"ig_(\d{4})_(\d{2})\.xlsx",
        nombre,
        re.IGNORECASE
    )

    fecha_archivo = None

    if coincidencia:
        anio = int(coincidencia.group(1))
        mes = int(coincidencia.group(2))

        fecha_archivo = datetime(
            anio, mes, 1
        ).date()

    registros_control.append((
        str(uuid.uuid4()),
        nombre,
        metadata["ruta"],
        fila["anio_origen"],
        fecha_archivo,
        metadata["tamanio"],
        fila["total_filas"],
        fila["total_columnas"],
        fila["total_celdas"],
        "COMPLETADO",
        fila["fecha_inicio"],
        fila["fecha_fin"],
        None
    ))

print("Registros de control preparados:", len(registros_control))

In [0]:
control_schema = StructType([
    StructField("id_archivo", StringType(), True),
    StructField("archivo_origen", StringType(), True),
    StructField("ruta_archivo", StringType(), True),
    StructField("anio_origen", IntegerType(), True),
    StructField("fecha_archivo", DateType(), True),
    StructField("tamanio_bytes", LongType(), True),
    StructField("total_filas", LongType(), True),
    StructField("total_columnas", IntegerType(), True),
    StructField("total_celdas", LongType(), True),
    StructField("estado_proceso", StringType(), True),
    StructField("fecha_inicio", TimestampType(), True),
    StructField("fecha_fin", TimestampType(), True),
    StructField("mensaje_error", StringType(), True)
])

df_control = spark.createDataFrame(
    registros_control,
    schema=control_schema
)

df_control.printSchema()

display(df_control)

In [0]:
tabla_delta = DeltaTable.forName(
    spark,
    tabla_control
)

(
    tabla_delta.alias("destino")
    .merge(
        df_control.alias("origen"),
        """
        destino.archivo_origen = origen.archivo_origen
        """
    )
    .whenMatchedUpdate(
        set={
            "ruta_archivo": "origen.ruta_archivo",
            "anio_origen": "origen.anio_origen",
            "fecha_archivo": "origen.fecha_archivo",
            "tamanio_bytes": "origen.tamanio_bytes",
            "total_filas": "origen.total_filas",
            "total_columnas": "origen.total_columnas",
            "total_celdas": "origen.total_celdas",
            "estado_proceso": "origen.estado_proceso",
            "fecha_inicio": "origen.fecha_inicio",
            "fecha_fin": "origen.fecha_fin",
            "mensaje_error": "origen.mensaje_error"
        }
    )
    .whenNotMatchedInsertAll()
    .execute()
)

print("Control de archivos actualizado correctamente")

In [0]:
%sql

SELECT
    archivo_origen,
    anio_origen,
    fecha_archivo,
    tamanio_bytes,
    total_filas,
    total_columnas,
    total_celdas,
    estado_proceso
FROM catalog_au.bronze.sfc_archivos_control
ORDER BY anio_origen;